# Validator Judge (LLM 3) — Proof of Concept

A first thin-slice prototype of the **correctness + uncertainty validator** from the design doc.

We **mock the upstream LLMs** (LLM 1 contextualization, LLM 1.2 comprehension, LLM 2 spoiler gate) and feed a fixed
*(reader question, source passage, generated answer)* into **LLM 3, the validator**.

The validator runs the full pipeline on one example:

1. **Decompose** the answer into atomic claims
2. **Route** each claim by grounding source (context / paraphrase / definition / world-knowledge)
3. **Verdict** per claim — Supported / Partially supported / Contradicted / Unverifiable (grounded against the passage)
4. **Aggregate** to an answer-level verdict (worst-case)
5. **Uncertainty** estimate
6. **Map** to a 3-way UI state — Valid / Not reliable / Hedged (Válido / Não confiável / Com ressalvas)

> **Note on uncertainty.** The doc's preferred "cheap single-pass logprob" signal is **not available** — the Anthropic API
> does not expose output-token logprobs. This PoC uses *verbalized confidence* (the doc's named baseline). *N-sample
> consistency* is the principled upgrade and is flagged where it slots in.

## 0. Dependencies

In [4]:
import subprocess, sys
pkgs = ["anthropic"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs)
print("Dependencies OK")

Dependencies OK


## 1. Configuration
Same backend as the anti-spoiler notebook — Anthropic + Haiku for cheap tokens.
Credentials load from Colab `userdata` if present, otherwise from the `ANTHROPIC_API_KEY` env var.

In [5]:
import os

# Credentials: Colab userdata -> env var fallback
API_KEY = None
try:
    from google.colab import userdata
    API_KEY = userdata.get("API_KEY")
except Exception:
    API_KEY = os.environ.get("ANTHROPIC_API_KEY")

# LLM backend
BACKEND         = "anthropic"
ANTHROPIC_MODEL = "claude-sonnet-4-6"  # judge: verdict stage is entailment reasoning -> stronger model helps
OPENAI_MODEL    = "gpt-4o"

print(f"Config  backend={BACKEND}  model={ANTHROPIC_MODEL}  key={'set' if API_KEY else 'MISSING'}")

Config  backend=anthropic  model=claude-sonnet-4-6  key=set


## 2. Model-agnostic LLM client + JSON helper
The same thin `LLMClient` wrapper from the anti-spoiler notebook, plus a small `parse_json_response`
helper that strips markdown fences — every validator stage returns JSON, so we factor that out once.

In [6]:
import json, re

class LLMClient:
    """Thin abstraction over Anthropic / OpenAI.  client.complete(system, user) -> str"""
    def __init__(self, backend: str, api_key: str | None = None):
        self.backend = backend.lower()
        if self.backend == "anthropic":
            import anthropic
            key = api_key or os.environ.get("ANTHROPIC_API_KEY")
            self._client = anthropic.Anthropic(api_key=key)
            self._model  = ANTHROPIC_MODEL
        elif self.backend == "openai":
            import openai
            key = api_key or os.environ.get("OPENAI_API_KEY")
            self._client = openai.OpenAI(api_key=key)
            self._model  = OPENAI_MODEL
        else:
            raise ValueError(f"Unknown backend {backend!r}. Choose anthropic | openai")

    def complete(self, system: str, user: str, max_tokens: int = 1024) -> str:
        if self.backend == "anthropic":
            msg = self._client.messages.create(
                model=self._model, max_tokens=max_tokens,
                system=system, messages=[{"role": "user", "content": user}],
            )
            return msg.content[0].text.strip()
        else:  # openai
            resp = self._client.chat.completions.create(
                model=self._model, max_tokens=max_tokens,
                messages=[{"role": "system", "content": system},
                          {"role": "user",   "content": user}],
            )
            return resp.choices[0].message.content.strip()


def parse_json_response(raw: str):
    """Strip ```fences``` and parse JSON; surfaces the raw text on failure."""
    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```[\w]*\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
    return json.loads(raw)


llm = LLMClient(backend=BACKEND, api_key=API_KEY)
print(f"LLMClient ready  ({BACKEND} / {llm._model})")

LLMClient ready  (anthropic / claude-sonnet-4-6)


## 3. Mock the upstream LLMs — contextualization path (LLM 1 → LLM 2 → LLM 3)

**Interaction model.** The reader never types a free-text question. They (1) **select a passage** and
(2) **click a feature button** — `Define` · `Paraphrase` · `Contextualize` · `Recall` — and the model
infers the request from *(selected text + feature)*. Selection is capped at the reader's position, so
nothing past their current chapter can be selected.

Our example uses **Contextualize** on a selected line about Mr. Bingley — a contextualization request,
so it runs LLM 1 → LLM 2 (spoiler gate) → the validator. We hard-code that path's *output* instead of
running it: the selected text, the feature, the in-bounds grounding passage, and the answer to validate.

The answer is **deliberately seeded** with a mix of claim types — i.e. we plant a known-wrong claim on
purpose (doc §8) so we can confirm the validator catches it:

| # | Claim | Grounding | Expected verdict |
|---|-------|-----------|------------------|
| 1 | Bingley is a single man of large fortune | context (passage) | Supported |
| 2 | He is already engaged to Jane | context (passage) | **Contradicted** (seeded error) |
| 3 | The novel was published in 1813 | world-knowledge | Unverifiable (no web search in PoC) |

Worst-case aggregation should therefore flag the whole answer as **Not reliable**.

In [7]:
# Mocked output of the contextualization path (LLM 1 -> LLM 2 spoiler gate -> validator).
# Interaction model: the reader does NOT type a question. They (1) select a passage and
# (2) click a feature button; the model infers the request from (selected text + feature).
# Feature buttons: "define" | "paraphrase" | "contextualize" | "recall".

READER_POSITION = 15   # reader is through chapter 15 of 61; grounding is limited to <= this

# What the reader actually did: highlighted a span + clicked a feature.
SELECTED_TEXT = "A single man of large fortune; four or five thousand a year. What a fine thing for our girls!"
FEATURE       = "contextualize"   # -> contextualization path (characters / places / background)

# Grounding context: stands in for the top-k relevant chunks retrieved from the reader's read-so-far
# scope (chapters <= READER_POSITION). In the final integrated validator this will come from bounded
# retrieval (retrieve_bounded + FAISS, as in the anti-spoiler notebook); we hardcoded here to isolate the judge (D15).
SOURCE_PASSAGE = """It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.

"My dear Mr. Bennet," said his lady to him one day, "have you heard that Netherfield Park is let at last?"

Mr. Bennet replied that he had not. "But it is," returned she; "for Mrs. Long has just been here, and she told me all about it."

"A single man of large fortune; four or five thousand a year. What a fine thing for our girls!"

"How so? how can it affect them?"

"My dear Mr. Bennet," replied his wife, "how can you be so tiresome! You must know that I am thinking of his marrying one of them."
"""

# Mocked answer the assistant produced for (SELECTED_TEXT, FEATURE), with a seeded error (claim 2).
GENERATED_ANSWER = (
    "Mrs. Bennet is excited because Mr. Bingley is a single man with a large fortune. "
    "She is especially pleased that he is already engaged to her eldest daughter, Jane. "
    "Pride and Prejudice was first published in 1813."
)

print(f"Reader position : through chapter {READER_POSITION} of 61")
print(f"Feature clicked : {FEATURE}")
print(f"Selected text   : {SELECTED_TEXT}")
print()
print("Generated answer to validate:")
print(" ", GENERATED_ANSWER)

Reader position : through chapter 15 of 61
Feature clicked : contextualize
Selected text   : A single man of large fortune; four or five thousand a year. What a fine thing for our girls!

Generated answer to validate:
  Mrs. Bennet is excited because Mr. Bingley is a single man with a large fortune. She is especially pleased that he is already engaged to her eldest daughter, Jane. Pride and Prejudice was first published in 1813.


## 4. Stage 4 — Decompose & route
Turn the free-text answer into a list of **atomic claims**, each tagged by **grounding source**
(`context` vs `world_knowledge`). No verdicts here — extraction + routing only (decision **D8**).
Rationale for the two-label set and the split-from-verdict choice: see `validator/DECISIONS.md`
(**D7**, **D8**).

**Contract:** `decompose_and_route(answer) -> [{"claim": str, "grounding": "context" | "world_knowledge"}, ...]`

In [8]:
SYSTEM_DECOMPOSE = """You are the claim-decomposition stage of a validation system for a reading assistant.

Your job: break an assistant's answer into ATOMIC CLAIMS and tag each by its GROUNDING SOURCE.
You do NOT judge whether claims are true. Extract faithfully: include every claim, even ones that
look wrong, and never correct, soften, or rephrase their meaning.

ATOMIC = one checkable fact per claim. Split conjunctions and compound sentences into separate
claims. Resolve pronouns and references so each claim stands on its own (e.g. "He" -> "Mr. Bingley").

GROUNDING SOURCE: tag each claim as exactly one of
- "context"         : a statement about what happens INSIDE the story (characters, events,
                      relationships, motivations, setting as described in the book). Must be checked
                      against the book passage the reader has read.
- "world_knowledge" : a statement about the real world or the book as a real-world artifact
                      (historical facts, publication dates, real people/places, literary background).
                      Would need an external source to verify.

Output ONLY a JSON array, no prose, no markdown fences. Each element:
  {"claim": "<the atomic claim as a standalone sentence>", "grounding": "context" | "world_knowledge"}
"""

def decompose_and_route(answer: str) -> list[dict]:
    """Answer text -> list of {"claim", "grounding"}. Routing only; no verdicts (D8)."""
    raw = llm.complete(SYSTEM_DECOMPOSE, f"ASSISTANT ANSWER TO DECOMPOSE:\n{answer}", max_tokens=600)
    return parse_json_response(raw)

In [9]:
claims = decompose_and_route(GENERATED_ANSWER)
print(f"{len(claims)} atomic claims:")
print()
for i, c in enumerate(claims, 1):
    print(f"{i}. [{c['grounding']:15}] {c['claim']}")

5 atomic claims:

1. [context        ] Mrs. Bennet is excited because Mr. Bingley is a single man.
2. [context        ] Mrs. Bennet is excited because Mr. Bingley has a large fortune.
3. [context        ] Mrs. Bennet is especially pleased that Mr. Bingley is already engaged to Jane.
4. [context        ] Jane is Mrs. Bennet's eldest daughter.
5. [world_knowledge] Pride and Prejudice was first published in 1813.
